In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display
import numpy as np
from PIL import Image
import io

In [ ]:
# Load video — update the path below
# VIDEO_PATH = "./supplementary/normalized/01_Anystyle_re10k_image_vs_text.mp4"
VIDEO_PATH = "./supplementary/normalized/02_Anystyle_re10k_natural_text_prompts.mp4"
# VIDEO_PATH = "./supplementary/normalized/03_Stylos_vs_Anystyle_comparison.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH}")
print(f"Frames: {total_frames}  |  FPS: {fps:.2f}  |  Resolution: {width}x{height}")

In [ ]:
def read_frame(frame_idx: int) -> Image.Image:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        raise ValueError(f"Could not read frame {frame_idx}")
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))


def frame_to_bytes(img: Image.Image) -> bytes:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return buf.getvalue()


# --- Widgets ---
slider = widgets.IntSlider(
    value=0, min=0, max=total_frames - 1, step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="90%"),
    style={"description_width": "60px"},
)
frame_label = widgets.Label(value=f"0 / {total_frames - 1}  |  t = 0.000 s")
img_widget = widgets.Image(format="jpeg", layout=widgets.Layout(max_width="100%"))

# Load first frame immediately
img_widget.value = frame_to_bytes(read_frame(0))


def on_slider_change(change):
    idx = change["new"]
    img_widget.value = frame_to_bytes(read_frame(idx))
    frame_label.value = f"{idx} / {total_frames - 1}  |  t = {idx / fps:.3f} s"


slider.observe(on_slider_change, names="value")

display(widgets.VBox([slider, frame_label, img_widget]))